# Avis Home — wake-word model training (openWakeWord, en_GB synthetic)

Trains 3 custom detectors — **Avis Home**, **Claude Fast**, **Claude Think**. Synthetic
positives use the British **en_GB-vctk-medium** voice (SA accent ~ British) and are folded
together with Jonathan & Chelsea's real voice clips; US adversarial negatives are generated
by openWakeWord as usual.

### How to run
1. **Runtime → Change runtime type → T4 GPU** (free).
2. **Runtime → Run all.**
3. Wait ~45-90 min (download + augmentation + training, x3, on Google's GPU).
4. The last cell downloads **`avis_wakeword_models.zip`** — send that back to Claude.

In [ ]:
# Environment setup (this cell takes a few minutes)
!git clone https://github.com/dscripka/piper-sample-generator
!wget -O piper-sample-generator/models/en-us-libritts-high.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'
# torch >=2.6 defaults weights_only=True, which rejects the pickled SynthesizerTrn checkpoint
!sed -i 's/model = torch.load(model_path)/model = torch.load(model_path, weights_only=False)/' piper-sample-generator/generate_samples.py
# the fork phonemizes via espeak_phonemizer (needs libespeak-ng), NOT piper-phonemize
!apt-get -qq install -y espeak-ng
!pip install espeak-phonemizer webrtcvad

!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

# torch 2.11's torchaudio dropped torchaudio.info, which torch-audiomentations' AddBackgroundNoise
# still calls -> shim it back (via soundfile) at the top of data.py so the augment subprocess has it
_p = "openwakeword/openwakeword/data.py"
_src = open(_p).read()
if "# --- torchaudio.info shim" not in _src:
    _shim = (
        '# --- torchaudio.info shim (torch 2.11 dropped torchaudio.info; torch-audiomentations needs it) ---\n'
        'import torchaudio as _ta_shim\n'
        'if not hasattr(_ta_shim, "info"):\n'
        '    from types import SimpleNamespace as _SNS\n'
        '    def _ta_info(uri, *a, **k):\n'
        '        import soundfile as _sf\n'
        '        _i = _sf.info(str(uri))\n'
        '        return _SNS(sample_rate=int(_i.samplerate), num_frames=int(_i.frames), num_channels=int(_i.channels), bits_per_sample=16, encoding="PCM_S")\n'
        '    _ta_shim.info = _ta_info\n'
        '# --- end shim ---\n'
    )
    open(_p, "w").write(_shim + _src)
    print("data.py torchaudio.info shim prepended")

# torch 2.11's torch.onnx.export defaults to the dynamo exporter (needs onnxscript, ignores
# opset_version, may externalise weights). Force the legacy TorchScript exporter so we get a
# self-contained opset-13 .onnx that the Pi's openWakeWord/onnxruntime loads.
!sed -i 's/opset_version=13)/opset_version=13, dynamo=False)/' openwakeword/openwakeword/train.py
!pip install onnxscript

# train.py also tries to convert the .onnx to TFLite (needs onnx_tf + tensorflow, which we omit).
# The .onnx is already written before this step, but the unguarded call raises ModuleNotFoundError.
# Neutralise just the call so the run ends cleanly (we only ship ONNX to the Pi).
_tp = "openwakeword/openwakeword/train.py"
_ts = open(_tp).read()
if 'lambda *a, **k: None)(os.path.join(config["output_dir"]' not in _ts:
    _ts = _ts.replace('convert_onnx_to_tflite(os.path.join(config["output_dir"]',
                      '(lambda *a, **k: None)(os.path.join(config["output_dir"]')
    open(_tp, "w").write(_ts)
    print("tflite conversion call neutralised (onnx_tf omitted; ONNX-only)")

!pip install mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14
# unpinned: the official ==0.11.0 pin fails to import on Colab's current torch
!pip install -U audiomentations torch-audiomentations acoustics
!pip install pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19
# NOTE: tensorflow-cpu==2.8.1 / tensorflow_probability / onnx_tf intentionally OMITTED
# (only needed for the optional TFLite export; we output ONNX, which the Pi's openWakeWord uses)

import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
base = "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1"
for m in ["embedding_model.onnx", "embedding_model.tflite", "melspectrogram.onnx", "melspectrogram.tflite"]:
    !wget -q {base}/{m} -O ./openwakeword/openwakeword/resources/models/{m}
print("install done")

In [ ]:
import os, sys, uuid, shutil, yaml, scipy.io.wavfile, numpy as np
from pathlib import Path
import datasets
from tqdm import tqdm

In [ ]:
# Room impulse responses (reverb augmentation)
os.makedirs("./mit_rirs", exist_ok=True)
rir = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir, desc="RIRs"):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join("./mit_rirs", name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
# Background noise via FMA, streamed -> 16k wav (same reliable datasets API as the RIRs;
# the AudioSet tar URL is broken on HF now)
os.makedirs("./background_clips", exist_ok=True)
if len(os.listdir("./background_clips")) < 200:
    fma = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    fma = iter(fma.cast_column("audio", datasets.Audio(sampling_rate=16000)))
    for i in tqdm(range(400), desc="fma->16k"):
        try:
            row = next(fma)
        except StopIteration:
            break
        arr = row['audio']['array']
        scipy.io.wavfile.write(os.path.join("./background_clips", f"fma_{i}.wav"),
                               16000, (arr * 32767).astype(np.int16))
print("background clips:", len(os.listdir("./background_clips")))

In [ ]:
# Pre-computed openWakeWord negative features (big download, reused for all 3 models)
import os
feat = "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
val = "validation_set_features.npy"
b = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main"
if not os.path.exists(feat) or os.path.getsize(feat) < 1e6:
    !wget -q {b}/{feat}
if not os.path.exists(val) or os.path.getsize(val) < 1e6:
    !wget -q {b}/{val}
print("features ready:", os.path.getsize(feat)//1024//1024, "MB +", os.path.getsize(val)//1024, "KB")

In [ ]:
# OPTIONAL: to fold in the real accent anchors, set REAL_REPO to a PRIVATE repo
# clone URL incl. token, e.g. "https://x-access-token:<TOKEN>@github.com/<you>/avis-wakeword-real-tmp.git"
# Leave it "" for an en_GB-synthetic-ONLY run (no personal data needed).
REAL_REPO = ""

In [ ]:
import glob, tarfile, os
SYNTH_REPO = "https://github.com/jonathanavis96/avis-wakeword-synth-tmp.git"
!rm -rf avis_repo
!git clone --depth 1 {SYNTH_REPO} avis_repo
tarfile.open("avis_repo/piper_positives.tar.gz").extractall("piper_positives_root")
PIPER_ROOT = "piper_positives_root/piper_positives"  # <model>/<split>/*.wav
print("piper en_GB positives:", len(glob.glob(PIPER_ROOT + "/**/*.wav", recursive=True)))
REAL_CLIPS_DIR = None
if REAL_REPO:
    !rm -rf real_repo && git clone --depth 1 {REAL_REPO} real_repo
    cand = "real_repo/clips_clean_train" if os.path.isdir("real_repo/clips_clean_train") else "real_repo"
    REAL_CLIPS_DIR = cand
    print("real clips:", len(glob.glob(REAL_CLIPS_DIR + "/**/*.wav", recursive=True)))
else:
    print("REAL_REPO empty -> en_GB-synthetic-ONLY run (no real clips folded in)")

In [ ]:
# Train all 3 wake-word models (reusing the negatives downloaded above)
PHRASES = [("Avis Home", "avis_home"), ("Claude Fast", "claude_fast"), ("Claude Think", "claude_think")]
SPEAKERS = ["jonathan", "chelsea"]

base_config = yaml.safe_load(open("openwakeword/examples/custom_model.yml").read())

def prefill_piper_positives(model_name):
    # Drop the locally-generated en_GB piper clips into positive_{train,test} so train.py's
    # --generate_clips sees positives already exist (>0.95*n_samples) and SKIPS its US synthesis,
    # while still generating the US adversarial negatives.
    out = os.path.abspath(base_config["output_dir"])
    pairs = [("train", os.path.join(out, model_name, "positive_train")),
             ("test",  os.path.join(out, model_name, "positive_test"))]
    n = 0
    for split, dst in pairs:
        os.makedirs(dst, exist_ok=True)
        for c in sorted(Path(f"{PIPER_ROOT}/{model_name}/{split}").glob("*.wav")):
            shutil.copy(c, os.path.join(dst, f"enGB_{c.stem}.wav")); n += 1
    print(f"  prefilled {n} en_GB piper positives for {model_name}")

def inject_real_clips(model_name):
    train_dir = os.path.join(base_config["output_dir"], model_name, "positive_train")
    test_dir = os.path.join(base_config["output_dir"], model_name, "positive_test")
    os.makedirs(train_dir, exist_ok=True); os.makedirs(test_dir, exist_ok=True)
    n = 0
    for sp in SPEAKERS:
        clips = sorted(Path(f"{REAL_CLIPS_DIR}/{sp}/{model_name}").glob("*.wav"))
        for i, c in enumerate(clips):
            dst = test_dir if i % 5 == 0 else train_dir   # ~20% to validation
            shutil.copy(c, os.path.join(dst, f"real_{sp}_{c.stem}.wav")); n += 1
    print(f"  injected {n} real clips for {model_name}")

for phrase, model_name in PHRASES:
    print(f"\n========== {phrase} ==========")
    cfg = dict(base_config)
    cfg["target_phrase"] = [phrase]
    cfg["model_name"] = model_name
    # n_samples / n_samples_val now size the (US) NEGATIVE generation + the positive-skip
    # threshold (0.95*n_samples). We prefill ~872 train / ~218 test en_GB positives, so keep
    # these below those counts to guarantee the US positive generation is skipped.
    cfg["n_samples"] = 800
    cfg["n_samples_val"] = 150
    cfg["steps"] = 10000
    cfg["target_accuracy"] = 0.6
    cfg["target_recall"] = 0.25
    cfg["background_paths"] = ["./background_clips"]
    cfg["false_positive_validation_data_path"] = "validation_set_features.npy"
    cfg["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
    yml = f"cfg_{model_name}.yaml"
    yaml.dump(cfg, open(yml, "w"))

    prefill_piper_positives(model_name)   # en_GB synthetic positives (skips US positive gen)
    if REAL_CLIPS_DIR:
        inject_real_clips(model_name)     # fold our accent anchors in before augmentation

    # --generate_clips now only generates the US adversarial NEGATIVES (positives already exist)
    !{sys.executable} openwakeword/openwakeword/train.py --training_config {yml} --generate_clips
    # --overwrite so augmentation always regenerates features (a prior crash can leave a partial
    # positive_features_train.npy that otherwise makes this step silently skip)
    !{sys.executable} openwakeword/openwakeword/train.py --training_config {yml} --augment_clips --overwrite
    !{sys.executable} openwakeword/openwakeword/train.py --training_config {yml} --train_model
print("\nALL MODELS TRAINED")

In [ ]:
# Package the trained ONNX models for download
import glob, zipfile, os
out = base_config["output_dir"]
# train.py may save the .onnx at out/ or out/<model_name>/ depending on version -> search both
models = sorted(set(glob.glob(os.path.join(out, "*.onnx")) +
                    glob.glob(os.path.join(out, "**", "*.onnx"), recursive=True)))
assert models, "No .onnx models found - check the training cell output above for errors."
with zipfile.ZipFile("avis_wakeword_models.zip", "w") as z:
    for m in models:
        z.write(m, os.path.basename(m))
print("packaged:", [os.path.basename(m) for m in models])
from google.colab import files
files.download("avis_wakeword_models.zip")